## 🔍 Diagnóstico Inicial: Análise Exploratória de Qualidade de Dados (EDA-DQ)

Antes de pensarmos em correções ou regras complexas, precisamos "sentir" os dados. Nesta primeira etapa, atuaremos como **Detetives de Dados**.

O objetivo não é gerar insights de negócio (como "qual produto vende mais"), mas sim realizar um **Check-up de Saúde do Dataset**. Assumimos a premissa de *Zero Trust*: não confiamos que os dados estão prontos para uso só porque o arquivo carregou sem erros.

### O Conceito: EDA voltada para Data Quality
Diferente da Análise Exploratória tradicional (focada em correlações e tendências), a **EDA-DQ** busca evidências de entropia e degradação:
* **Completo?** Existem "buracos" (nulos) em campos críticos?
* **Único?** Existem entidades duplicadas inflando os números?
* **Consistente?** A distribuição estatística faz sentido (ex: preços negativos)?
* **Íntegro?** Os tipos de dados (Schema) estão respeitando o formato esperado?

#### Visão Geral e Schema (Overview)

In [1]:
import pandas as pd

In [2]:
df_vendas = pd.read_csv('./data/sales.csv', index_col=None)

In [3]:
# 1. Visão Geral (Amostra e Tipos)
print("--- Dimensões do Dataset ---")
print(f"Linhas: {df_vendas.shape[0]}")
print(f"Colunas: {df_vendas.shape[1]}")

print("\n--- Amostra das primeiras 5 linhas ---")
display(df_vendas.head())

print("\n--- Amostra das últimas 5 linhas ---")
display(df_vendas.tail())

print("\n--- Schema e Tipos de Dados (Inferidos) ---")
df_vendas.info()

--- Dimensões do Dataset ---
Linhas: 2050
Colunas: 11

--- Amostra das primeiras 5 linhas ---


,transaction_id,customer_id,customer_name,customer_email,transaction_date,category,unit_price,quantity,total_amount,payment_method,status
0,26ef9d11-367c-4c03-94c3-5d52d8e2463f,5506.0,Alice Viana,wdias@example.org,2099-12-31,Livros,3200.74,1,3200.74,Pix,Aprovado
1,e3ed2e31-19f4-48a4-8902-a976a331f010,9935.0,Vitor Rezende,amanda00@example.org,2099-12-31,Eletrônicos,706.29,1,706.29,Voucher,Aprovado
2,9b5976e0-3d20-4379-87ed-9b3bdd693ad3,4582.0,Srta. Alice da Rosa,gnascimento@example.net,2099-12-31,Livros,168.60,1,168.60,Cartão de Crédito,Aprovado
3,666ae62d-1b21-497b-92e5-059f56b71d0e,7873.0,Enzo Fonseca,monteirosofia@example.com,2099-12-31,Livros,2810.61,5,14053.05,Voucher,Aprovado
4,82bf8067-011c-4104-83e8-ae97c2900802,3615.0,Maya Mendes,da-rosaeduardo@example.org,2099-12-31,Moda,-2950.44,1,2950.44,Boleto,Aprovado



--- Amostra das últimas 5 linhas ---


,transaction_id,customer_id,customer_name,customer_email,transaction_date,category,unit_price,quantity,total_amount,payment_method,status
2045,64189c6b-d2a0-4001-b52e-4d9fa6cdcee8,8260.0,Dr. Luiz Fernando Nascimento,wfreitas@example.net,2025-03-08,Brinquedos,2318.51,1,2318.51,Cartão de Crédito,Aprovado
2046,b60ad4ac-4f6d-4f0b-aa73-357572a032f5,1241.0,Sr. Luiz Gustavo da Costa,NaN,2025-07-12,Eletro,262.44,5,1312.20,Pix,Aprovado
2047,594bd1f4-0074-487d-a6e9-715e54fd0abc,8886.0,Lunna Almeida,iferreira@example.com,2025-10-03,Livros,839.89,4,3359.56,Voucher,Aprovado
2048,b0ae23f4-9d4d-4844-a0a2-1f30da6382dd,7209.0,Vicente Gonçalves,dmelo@example.net,2025-06-29,Eletrônicos,4513.19,2,9026.38,Voucher,Aprovado
2049,ac576e88-94b7-4600-9585-35abd0dfacf5,5673.0,Luiz Otávio Jesus,NaN,2025-10-02,Moda,1333.36,4,5333.44,Voucher,Aprovado



--- Schema e Tipos de Dados (Inferidos) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2050 entries, 0 to 2049
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    2050 non-null   object 
 1   customer_id       1943 non-null   float64
 2   customer_name     2050 non-null   object 
 3   customer_email    1865 non-null   object 
 4   transaction_date  2050 non-null   object 
 5   category          2050 non-null   object 
 6   unit_price        2050 non-null   float64
 7   quantity          2050 non-null   object 
 8   total_amount      2050 non-null   float64
 9   payment_method    2050 non-null   object 
 10  status            2050 non-null   object 
dtypes: float64(3), object(8)
memory usage: 176.3+ KB


#### Análise de Completude (Completeness)

In [4]:
# 2. Análise de Nulos (Completeness)
total_nulos = df_vendas.isnull().sum()
percentual_nulos = (df_vendas.isnull().sum() / len(df_vendas)) * 100

# Criando um DataFrame para facilitar a visualização
df_nulos = pd.DataFrame({
    'Total Nulos': total_nulos,
    'Percentual (%)': percentual_nulos
})

# Filtrando apenas colunas que têm problemas
print("--- Relatório de Completude (Colunas com dados faltantes) ---")
display(df_nulos[df_nulos['Total Nulos'] > 0].sort_values(by='Total Nulos', ascending=False))

--- Relatório de Completude (Colunas com dados faltantes) ---


,Total Nulos,Percentual (%)
customer_email,185,9.024390
customer_id,107,5.219512


#### Análise de Unicidade (Uniqueness)

In [5]:
# 3. Análise de Duplicidade

# A. Linhas completamente duplicadas (Registro inteiro repetido)
duplicadas_totais = df_vendas.duplicated().sum()
print(f"Linhas totalmente duplicadas: {duplicadas_totais}")

# B. Violação de Chave Primária (IDs repetidos com dados diferentes ou iguais)
# Supondo que 'transaction_id' deve ser único por natureza
ids_duplicados = df_vendas[df_vendas.duplicated(subset=['transaction_id'], keep=False)]

print(f"\nViolações de Unicidade (Transaction IDs repetidos): {ids_duplicados['transaction_id'].nunique()} IDs únicos afetados.")

if not ids_duplicados.empty:
    print("Exemplo de ID duplicado:")
    # Pega o primeiro ID duplicado para mostrar
    exemplo_id = ids_duplicados['transaction_id'].iloc[0]
    display(df_vendas[df_vendas['transaction_id'] == exemplo_id])

Linhas totalmente duplicadas: 19

Violações de Unicidade (Transaction IDs repetidos): 50 IDs únicos afetados.
Exemplo de ID duplicado:


,transaction_id,customer_id,customer_name,customer_email,transaction_date,category,unit_price,quantity,total_amount,payment_method,status
0,26ef9d11-367c-4c03-94c3-5d52d8e2463f,5506.0,Alice Viana,wdias@example.org,2099-12-31,Livros,3200.74,1,3200.74,Pix,Aprovado
2000,26ef9d11-367c-4c03-94c3-5d52d8e2463f,5506.0,Alice Viana,wdias@example.org,2025-02-14,Livros,3200.74,1,3200.74,Pix,Aprovado


#### Análise de Consistência Numérica (Validity & Range)

In [6]:
# 4. Estatística Descritiva (Detectando Outliers e Valores Negativos)

cols_numericas = ['unit_price', 'quantity', 'total_amount']
temp_desc = df_vendas.copy()

for col in cols_numericas:
    # errors='coerce' vai transformar textos como "dois" em NaN temporariamente
    temp_desc[col] = pd.to_numeric(temp_desc[col], errors='coerce')

print("--- Resumo Estatístico ---")
display(temp_desc[cols_numericas].describe().round(2))

--- Resumo Estatístico ---


,unit_price,quantity,total_amount
count,2050.00,2048.0,2050.00
mean,2478.83,3.0,7286.38
std,1496.40,1.4,5999.32
min,-4544.69,1.0,17.89
25%,1202.97,2.0,2312.67
50%,2453.15,3.0,5346.23
75%,3774.42,4.0,10998.53
max,4999.54,5.0,24889.25


#### Análise de Domínio Categórico (Conformity)

In [8]:
# 5. Análise de Frequência (Categorias e Status)

print("--- Inconsistências na Coluna 'Category' ---")
# Mostra todas as variações únicas. O aluno verá "Eletrônicos", "eletronicos", "Eletro"
print(df_vendas['category'].unique())

print("\n--- Contagem de Valores (Top 10) ---")
print(df_vendas['category'].value_counts().head(10))

--- Inconsistências na Coluna 'Category' ---
['Livros' 'Eletrônicos' 'Moda' 'Brinquedos' 'Casa' 'casa' 'CASA'
 'eletronicos' 'ELETRONICOS ' 'Eletro ' 'Home']

--- Contagem de Valores (Top 10) ---
category
Moda            435
Brinquedos      410
Livros          407
Eletrônicos     317
Casa            312
ELETRONICOS      35
CASA             32
Eletro           30
casa             29
eletronicos      23
Name: count, dtype: int64


#### Investigação Detalhada das Anomalias

In [ ]:
print("1. O Mistério do Preço Negativo:")
# Filtrando onde o preço é menor que zero
precos_errados = df_vendas[ pd.to_numeric(df_vendas['unit_price'], errors='coerce') < 0 ]
display(precos_errados[['transaction_id', 'unit_price', 'quantity', 'total_amount']])

print("\n2. O Mistério da 'Quantidade' que sumiu (Erro de Tipo):")
qtd_invalida = df_vendas[ 
    pd.to_numeric(df_vendas['quantity'], errors='coerce').isna() & 
    df_vendas['quantity'].notna() 
]

qtd_invalida[['transaction_id', 'quantity', 'category']].head()

1. O Mistério do Preço Negativo:


,transaction_id,unit_price,quantity,total_amount
4,82bf8067-011c-4104-83e8-ae97c2900802,-2950.44,1,2950.44
142,d7dd2f29-b221-4320-8164-f2bdadf6eedc,-521.91,1,521.91
281,ffb328f9-6690-481f-a31c-552b0487bfad,-1246.24,4,4984.96
525,501db5bf-c0bf-4889-82a8-eeb05f477080,-155.67,5,778.35
929,9bfc77ea-bc04-449f-8c8c-090ad1d971c7,-4072.81,3,12218.43
1318,8566a5b2-0178-42d8-880e-7b14f207c7a1,-905.07,2,1810.14
1712,c3fb4049-1988-42ec-aeeb-616750abb6e4,-4544.69,2,9089.38
1838,94efebaa-0723-4942-8ada-a2eb29dcfce6,-362.08,3,1086.24
1895,41984108-38a5-4b81-be5a-3a9af367ab43,-3305.55,3,9916.65
1967,e60fe111-4381-4475-8e91-1428e578f0ad,-2063.50,3,6190.50



2. O Mistério da 'Quantidade' que sumiu (Erro de Tipo):


,transaction_id,quantity,category
100,559652ba-1083-4bbf-8d42-3d5f9eed1fc7,dois,eletronicos
101,16dc1141-0922-4f57-8dd6-25009beea996,10 un.,Brinquedos
